# Stage 2 Colab A100 executor
Upload `/content/stage2_source.zip` and its externally generated `/content/stage2_source_expectations.json` sidecar.

In [ ]:
import hashlib, json, re, shutil, subprocess, zipfile
from pathlib import Path, PurePosixPath
content=Path('/content'); source_zip=content/'stage2_source.zip'; expectations=content/'stage2_source_expectations.json'
expected=json.loads(expectations.read_text(encoding='utf-8'))
required={'schema_version','archive_sha256','origin_commit','execution_commit','source_manifest_sha256'}
hex40=re.compile(r'^[0-9a-f]{40}$'); hex64=re.compile(r'^[0-9a-f]{64}$')
if set(expected)!=required or expected['schema_version']!='stage2_source_expectations_v1' or not hex64.fullmatch(expected['archive_sha256']) or not hex40.fullmatch(expected['origin_commit']) or not hex40.fullmatch(expected['execution_commit']) or not hex64.fullmatch(expected['source_manifest_sha256']): raise RuntimeError('invalid external source expectations')
if hashlib.sha256(source_zip.read_bytes()).hexdigest()!=expected['archive_sha256']: raise RuntimeError('outer source ZIP checksum mismatch')
unpack=content/'stage2_source_unpack'; repo=content/'stage2_repo'; verify=content/'stage2_bundle_verify'
for path in (unpack,repo,verify):
    if path.exists(): shutil.rmtree(path)
with zipfile.ZipFile(source_zip) as z:
    names=z.namelist()
    if set(names)!={'stage2_source.bundle','source_metadata.json'} or len(names)!=len(set(names)) or any(PurePosixPath(n).is_absolute() or '..' in PurePosixPath(n).parts or '\\' in n for n in names): raise RuntimeError('unsafe source archive')
    unpack.mkdir(); [(unpack/n).write_bytes(z.read(n)) for n in names]
meta=json.loads((unpack/'source_metadata.json').read_text(encoding='utf-8')); bundle=unpack/'stage2_source.bundle'; manifest=meta.get('source_manifest')
canonical=lambda value: json.dumps(value,ensure_ascii=False,allow_nan=False,sort_keys=True,separators=(',',':')).encode('utf-8')
meta_keys={'schema_version','git','protocol_path','protocol_sha256','bundle_sha256','source_manifest','source_manifest_sha256','exclusions'}; git_keys={'origin_commit','execution_commit','branch','dirty','status_porcelain'}
if set(meta)!=meta_keys or meta.get('schema_version')!='stage2_source_package_v2' or not isinstance(meta.get('git'),dict) or set(meta['git'])!=git_keys or meta['git'].get('dirty') is not False or meta['git'].get('status_porcelain')!=[] or meta['git'].get('origin_commit')!=expected['origin_commit'] or meta['git'].get('execution_commit')!=expected['execution_commit'] or meta.get('source_manifest_sha256')!=expected['source_manifest_sha256'] or hashlib.sha256(bundle.read_bytes()).hexdigest()!=meta.get('bundle_sha256') or not isinstance(manifest,list) or hashlib.sha256(canonical(manifest)).hexdigest()!=expected['source_manifest_sha256']: raise RuntimeError('source metadata mismatch')
allowed_dirs={'canonical','configs','data','docs','models','notebooks','tests','training','utils'}; root_files={'.gitignore','LICENSE','requirements.txt'}; excluded={'ties_results','ties_unlearn_results','.venv-stage2','.uv-cache','__pycache__','.stage2_monitor','.git','.worktrees','out'}; forbidden_suffixes={'.zip','.pt','.pth','.ckpt','.bin','.safetensors','.pyc'}
def allowed_source(name):
    path=PurePosixPath(name); safe=bool(name) and not path.is_absolute() and '..' not in path.parts and '\\' not in name and all(part not in {'','.'} for part in path.parts)
    return safe and not any(part.casefold() in excluded for part in path.parts) and path.suffix.casefold() not in forbidden_suffixes and ((len(path.parts)>1 and path.parts[0] in allowed_dirs) or (len(path.parts)==1 and (name in root_files or path.suffix in {'.py','.md'})))
if any(not isinstance(e,dict) or set(e)!={'path','mode','blob_sha1','content_sha256'} or not hex40.fullmatch(str(e.get('blob_sha1',''))) or not hex64.fullmatch(str(e.get('content_sha256',''))) or e.get('mode') not in {'100644','100755'} or not allowed_source(str(e.get('path',''))) for e in manifest): raise RuntimeError('unsafe source manifest')
if manifest!=sorted(manifest,key=lambda e:e['path'].encode('utf-8')) or len({e['path'] for e in manifest})!=len(manifest) or not isinstance(meta.get('protocol_path'),str) or not any(e['path']==meta['protocol_path'] and e['content_sha256']==meta.get('protocol_sha256') for e in manifest): raise RuntimeError('source manifest ordering/protocol mismatch')
verify.mkdir(); subprocess.run(['git','init','-q'],check=True,cwd=verify)
subprocess.run(['git','bundle','verify',str(bundle)],check=True,cwd=verify)
heads=subprocess.run(['git','bundle','list-heads',str(bundle)],check=True,cwd=verify,capture_output=True,text=True).stdout.strip().splitlines()
if heads!=[expected['execution_commit']+' refs/heads/stage2-execution']: raise RuntimeError('bundle ref mismatch')
subprocess.run(['git','-c','core.autocrlf=false','clone',str(bundle),str(repo)],check=True,cwd=content)
subprocess.run(['git','config','core.autocrlf','false'],check=True,cwd=repo)
subprocess.run(['git','checkout','--detach',expected['execution_commit']],check=True,cwd=repo)
head=subprocess.run(['git','rev-parse','HEAD'],check=True,cwd=repo,capture_output=True,text=True).stdout.strip()
parents=subprocess.run(['git','rev-list','--parents','-n','1','HEAD'],check=True,cwd=repo,capture_output=True,text=True).stdout.split()
tree_raw=subprocess.run(['git','ls-tree','-r','-z','HEAD'],check=True,cwd=repo,capture_output=True).stdout; actual=[]
for record in tree_raw.split(b'\0'):
    if not record: continue
    header,name=record.split(b'\t',1); mode,kind,blob=header.decode().split(); path=name.decode()
    payload=subprocess.run(['git','cat-file','blob',blob],check=True,cwd=repo,capture_output=True).stdout
    actual.append({'path':path,'mode':mode,'blob_sha1':blob,'content_sha256':hashlib.sha256(payload).hexdigest()})
status=subprocess.run(['git','status','--porcelain'],check=True,cwd=repo,capture_output=True,text=True).stdout.strip()
if head!=expected['execution_commit'] or parents!=[head] or actual!=manifest or status: raise RuntimeError('execution checkout provenance mismatch')
gpu=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],check=True,cwd=repo,capture_output=True,text=True).stdout.strip().splitlines()[0]
if 'A100' not in gpu: raise RuntimeError(f'Stage 2 requires A100, found {gpu}')

In [ ]:
subprocess.run(['python','-m','pip','install','--upgrade','pip'],check=True,cwd=repo)
subprocess.run(['python','-m','pip','install','torch==2.11.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True,cwd=repo)
torch_gpu=subprocess.run(['python','-c',"import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"],check=True,cwd=repo,capture_output=True,text=True).stdout.strip()
if torch_gpu!=gpu or 'A100' not in torch_gpu: raise RuntimeError(f'torch/nvidia-smi GPU mismatch: {torch_gpu!r} != {gpu!r}')
subprocess.run(['python','-m','pip','install','-r','requirements.txt'],check=True,cwd=repo)
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True,cwd=repo)
primary='ties_results/stage2_smoke/colab_a100_run1'; repeat='ties_results/stage2_smoke/colab_a100_repeat_full_sr'; canonical='ties_results/canonical_v1'
subprocess.run(['python','monitor_stage2_job.py','--events','ties_results/.stage2_monitor/colab_a100_run1.events.jsonl','--watch',primary,'--','python','run_stage2_smoke.py','--mode','primary','--environment','colab_a100','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--output-dir',primary,'--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical],check=True,cwd=repo)
subprocess.run(['python','monitor_stage2_job.py','--events','ties_results/.stage2_monitor/colab_a100_repeat_full_sr.events.jsonl','--watch',repeat,'--','python','run_stage2_smoke.py','--mode','repeat_full_sr','--environment','colab_a100','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--output-dir',repeat,'--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical,'--compare-repeat',repeat],check=True,cwd=repo)
subprocess.run(['python','freeze_stage2_environment.py','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--smoke-root',primary,'--repeat-root',repeat,'--source-archive','/content/stage2_source.zip','--source-expectations','/content/stage2_source_expectations.json','--commands',primary+'/commands.json','--repeat-commands',repeat+'/commands.json','--repo-root','.','--output-dir','ties_results/stage2_smoke/freeze_bundle','--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical,'--compare-repeat',repeat],check=True,cwd=repo)
subprocess.run(['python','freeze_stage2_environment.py','--verify-only','--output-dir','ties_results/stage2_smoke/freeze_bundle'],check=True,cwd=repo)
subprocess.run(['python','package_stage2_evidence.py','--repo-root','.','--output','/content/stage2_a100_evidence.zip','--source-expectations','/content/stage2_source_expectations.json'],check=True,cwd=repo)